<a href="https://colab.research.google.com/github/Evangel90/Attendance-Portal/blob/master/WhisperFinetunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparing environment

In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Tue Jun 17 20:23:30 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install git+https://github.com/openai/whisper.git
!pip install transformers datasets torchaudio librosa jiwer accelerate

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-yik3mi3d
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-yik3mi3d
  Resolved https://github.com/openai/whisper.git to commit dd985ac4b90cafeef8712f2998d62c59c3e62d22
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from huggingface_hub import notebook_login

notebook_login()

# Load Dataset

In [4]:
!git lfs install
!git clone https://huggingface.co/datasets/benjaminogbonna/nigerian_accented_english_dataset


Git LFS initialized.
fatal: destination path 'nigerian_accented_english_dataset' already exists and is not an empty directory.


In [5]:
!pip install pandas pyarrow datasets


In [6]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Replace with your actual path
DATA_DIR = "./nigerian_accented_english_dataset/data"

train_df = pd.read_parquet(f"{DATA_DIR}/train-00000-of-00001.parquet")
val_df = pd.read_parquet(f"{DATA_DIR}/validation-00000-of-00001.parquet")
test_df = pd.read_parquet(f"{DATA_DIR}/test-00000-of-00001.parquet")

In [7]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'client_id', 'path', 'sentence', 'accent', 'locale', 'segment'],
        num_rows: 2721
    })
    validation: Dataset({
        features: ['audio', 'client_id', 'path', 'sentence', 'accent', 'locale', 'segment'],
        num_rows: 340
    })
    test: Dataset({
        features: ['audio', 'client_id', 'path', 'sentence', 'accent', 'locale', 'segment'],
        num_rows: 341
    })
})


In [8]:
# Keep only the 'audio' and 'sentence' columns in each split
dataset = dataset.remove_columns([col for col in dataset["train"].column_names if col not in ["audio", "sentence"]])

print(dataset)
print(dataset["train"][0])


DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 2721
    })
    validation: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 340
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 341
    })
})
{'audio': {'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x00\x00\x00\x0f\x00\x00\x03Lavf58.29.100\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xf3X\xc0\x00\x00\x00\x00\x00\x00\x00\x00\x00Info\x00\x00\x00\x0f\x00\x00\x00\x98\x00\x00@\xd4\x00\x06\t\x0b\x0e\x10\x13\x15\x18\x1a\x1d\x1f"$\'),.1368;=@CEHJMORTWY\\^acfhkmpruwz|\x7f\x83\x84\x88\x89\x8d\x8e\x92\x93\x97\x98\x9c\x9d\xa1\xa2\xa6\xa7\xab\xac\xb0\xb1\xb5\xb6\xba\xbb\xbf\xc2\xc4\xc7\xc9\xcc\xce\xd1\xd3\xd6\xd8\xdb\xdd\xe0\xe2\xe5\xe7\xea\xec\xef\xf1\xf4\xf6\xf9\xfb\xfe\x00\x00\x00\x00Lavc58.54\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00$\x03\xc0\x00\x00\x00\x00\x00\x00@\xd4O\xbe!\xf2\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xf38\xc4\

# Resample audio and prepossesing

In [9]:
!pip install torchaudio soundfile


In [13]:
import io
import torchaudio
from datasets import Audio
from transformers import WhisperProcessor, WhisperTokenizer

# Assuming processor and tokenizer are already loaded from previous cells
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny")

# Cast the audio column to the Audio feature. This handles loading and setting sampling rate.
# The target sampling rate for Whisper is 16000 Hz.
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

print("Dataset after casting audio column to Audio feature:")
print(dataset)
print("Example from train split after casting audio column:")
print(dataset["train"][0])



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset after casting audio column to Audio feature:
DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 2721
    })
    validation: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 340
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 341
    })
})
Example from train split after casting audio column:
{'audio': {'path': 'audio_sample_10.mp3', 'array': array([ 1.37660527e-05,  1.56556907e-05,  6.42209125e-06, ...,
       -6.02160653e-05, -8.55889266e-06,  5.51452322e-05]), 'sampling_rate': 16000}, 'sentence': 'Closing the Google assistant app prevents it from working with your headphones.'}


In [14]:
# Define a single function to prepare the example for training
def prepare_training_example(example):
    # The Audio feature loads and resamples the audio to 16kHz automatically
    # The audio is now available as a numpy array under 'audio["array"]'
    # and the sampling rate under 'audio["sampling_rate"]'
    audio = example["audio"]

    # Use the processor's feature extractor to get log-mel spectrogram features
    example["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Use the processor's tokenizer to get the tokenized label ids from the sentence
    # Ensure 'sentence' exists and is not None or empty
    sentence = example["sentence"]
    if not isinstance(sentence, str) or not sentence.strip():
        # Handle cases with missing or empty sentences if necessary
        # For training, it's often better to filter these examples out earlier
        print(f"Warning: Skipping tokenization for example with missing/empty sentence.")
        example["labels"] = [] # Or handle as an error/filter
    else:
        example["labels"] = processor.tokenizer(sentence).input_ids

    return example

# Apply this combined function to all splits and remove original columns
# The `remove_columns` argument in .map() is the correct place to drop columns
dataset = dataset.map(
    prepare_training_example,
    remove_columns=["audio", "sentence"],
    num_proc=1, # Use num_proc=1 for easier debugging if issues persist
    batched=False # Set to False as we are processing one example at a time
)

print("\nDataset after preparing training examples:")
print(dataset)
print("Example from train split after preparing training examples:")
print(dataset["train"][0])


Map:   0%|          | 0/2721 [00:00<?, ? examples/s]

Map:   0%|          | 0/340 [00:00<?, ? examples/s]

Map:   0%|          | 0/341 [00:00<?, ? examples/s]


Dataset after preparing training examples:
DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 2721
    })
    validation: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 340
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 341
    })
})
Example from train split after preparing training examples:
{'input_features': [[-0.5039801597595215, -0.5039801597595215, -0.3799244165420532, -0.5039801597595215, -0.3594334125518799, -0.5039801597595215, -0.4867299795150757, -0.34252071380615234, -0.3679211139678955, -0.360587477684021, -0.21225571632385254, -0.3597975969314575, -0.20630085468292236, -0.2107638120651245, -0.37358903884887695, -0.3616436719894409, -0.464762806892395, -0.5039801597595215, -0.37825703620910645, -0.5039801597595215, -0.5039801597595215, -0.4496387243270874, -0.5039801597595215, -0.48551857471466064, -0.5039801597595215, -0.4855618476867676, -0.452568

In [15]:
# Add the check for required keys after the final mapping
required_keys = ["input_features", "labels"] # input_features for the data collator, labels for training

print("\nChecking for required keys in the dataset splits before training...")
for split_name, split_dataset in dataset.items():
    first_example_keys = split_dataset[0].keys()
    missing = [key for key in required_keys if key not in first_example_keys]
    if missing:
        print(f"Split '{split_name}' is missing required keys: {missing}")
        print(f"Keys present in {split_name}[0]: {first_example_keys}")
        # Consider raising an error here if keys are missing
        raise ValueError(f"Dataset split '{split_name}' is missing required keys.")
    else:
         print(f"Split '{split_name}' has all required keys.")


Checking for required keys in the dataset splits before training...
Split 'train' has all required keys.
Split 'validation' has all required keys.
Split 'test' has all required keys.


# Training

## Load Pretrained Whisper Tiny and Processor

In [16]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")

# Force the model to use English and transcribe task
model.generation_config.language = "english"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None


## Define Data Collator

In [17]:
# The DataCollator and Trainer code remain the same

import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Audio inputs
        input_features = [f["input_features"] for f in features]
        batch = {"input_features": torch.tensor(input_features)}

        # Labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id
)

## Define WER Metric

In [18]:
!pip install evaluate

In [19]:
import evaluate
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {"wer": metric.compute(predictions=pred_str, references=label_str)}


## Set Training Arguments

In [20]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny-en-nigeria",
    per_device_train_batch_size=1,  # low batch size due to CPU
    learning_rate=1e-5,
    max_steps=500,  # small steps due to CPU limits
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,
    predict_with_generate=True,
    generation_max_length=225,
    push_to_hub=False,  # disable if not uploading
)


## Initialize and Start Training

In [21]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor,
)


# Save processor config
processor.save_pretrained(training_args.output_dir)

# Start training
trainer.train()

<ipython-input-21-2117991498>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: evangel-m1902505 (evangel-m1902505-federal-university-of-technology-minna) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss,Wer
100,1.670800,2.002006,0.608345
200,1.726700,1.847548,0.644439
300,1.651800,1.767152,0.600026
400,1.897800,1.731296,0.593626
500,1.673000,1.719662,0.611417


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 2

TrainOutput(global_step=500, training_loss=1.807550832748413, metrics={'train_runtime': 821.789, 'train_samples_per_second': 0.608, 'train_steps_per_second': 0.608, 'total_flos': 1.230944256e+16, 'train_loss': 1.807550832748413, 'epoch': 0.18375597206909225})

## Test

In [22]:
metrics = trainer.evaluate(dataset["test"])
print(metrics)

{'eval_loss': 1.6824116706848145, 'eval_wer': 0.6617946110828673, 'eval_runtime': 117.3722, 'eval_samples_per_second': 2.905, 'eval_steps_per_second': 0.366, 'epoch': 0.18375597206909225}


### Inference on sample audio

In [25]:
import torchaudio
from datasets import Audio # Import Audio feature

# Load audio file using datasets.Audio to handle resampling
# Create a dummy Dataset object to use the Audio feature for loading/resampling
from datasets import Dataset

# Define a dummy feature structure to use the Audio feature
features = Dataset.from_dict({"audio": ["/content/test_audio.wav"]}).features

# Cast the 'audio' column to the Audio feature with the target sampling rate
# This will load and resample the audio when the example is accessed
features["audio"] = Audio(sampling_rate=16000)

# Create a dummy dataset with the file path and the defined features
dummy_dataset = Dataset.from_dict({"audio": ["/content/test_audio.wav"]}).cast_column("audio", Audio(sampling_rate=16000))

# Access the loaded and resampled audio from the dummy dataset
# This gives you the audio array and the (now corrected) sampling rate
audio_data = dummy_dataset[0]["audio"]
audio_array = audio_data["array"]
sample_rate = audio_data["sampling_rate"] # This will now be 16000

# Now, preprocess with the processor using the correctly sampled audio
# The processor will receive audio_array (1D numpy array) and sample_rate (16000)
input_features = processor(audio_array, sampling_rate=sample_rate, return_tensors="pt").input_features

# Move the input features to the same device as the model (GPU)
input_features = input_features.to(model.device)

# Generate
predicted_ids = model.generate(input_features)
transcription = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print("Transcription:", transcription)

Transcription:  Who is the HOD of computer engineering department, the University of Technology, Nina?
